In [ ]:
import cv2
import os

# input folder
frames_folder = "segmentation/gray"   

# output folder
output_video_path = "tracked_output.mp4"

os.makedirs("tracked_frames", exist_ok=True)

# Get frames
frames = sorted([
    f for f in os.listdir(frames_folder)
    if f.lower().endswith((".jpg", ".png"))
])

if len(frames) == 0:
    print("No frames found!")
    exit()

# Read first frame
first_frame = cv2.imread(os.path.join(frames_folder, frames[0]))

if first_frame is None:
    print("Cannot read first frame")
    exit()

print("Select 3 players")

bboxes = []
for i in range(3):
    bbox = cv2.selectROI("Select Player", first_frame, False)
    if bbox[2] > 0 and bbox[3] > 0:
        bboxes.append(bbox)

cv2.destroyAllWindows()

# Create trackers
trackers = []
for bbox in bboxes:
    tracker = cv2.TrackerCSRT_create()
    tracker.init(first_frame, bbox)
    trackers.append(tracker)

# Video writer setup
h, w = first_frame.shape[:2]
fps = 25
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

# =========================
# TRACKING LOOP
# =========================
for frame_name in frames:

    frame_path = os.path.join(frames_folder, frame_name)
    frame = cv2.imread(frame_path)

    if frame is None:
        continue

    # Update each tracker
    for i, tracker in enumerate(trackers):

        success, bbox = tracker.update(frame)

        if success:
            x, y, w_box, h_box = [int(v) for v in bbox]

            cv2.rectangle(frame,
                          (x, y),
                          (x + w_box, y + h_box),
                          (0, 255, 0), 2)

            cv2.putText(frame,
                        f"Player {i+1}",
                        (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2)

    # Save video frame
    out.write(frame)

    # Show live tracking
    cv2.imshow("Tracking", frame)

    if cv2.waitKey(30) & 0xFF == 27:
        break

out.release()
cv2.destroyAllWindows()

print("Video saved:", output_video_path)

Select 3 players
